In [1]:
### 逐帧分析细胞轨迹动态信息 ####
### 20260622 ####
### cDC_CD8Tn & cDC_CD4Tn ###


In [ ]:
# =============================================================================
# Napari细胞轨迹可视化 + 单细胞运动数据导出 (批量处理) - V4
# 关键：
#   1. 使用实际拍摄间隔（60秒/时间点）
#   2. Burst模式：每9帧抽取1帧
# CD4: kernel=61, CD8: kernel=101, cDC: kernel=61
# =============================================================================

import os
import glob
import cv2
import numpy as np
import pandas as pd
import trackpy as tp
from skimage import measure, morphology
import napari
from tqdm import tqdm
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# 全局配置
# =============================================================================
PIXEL_SIZE_UM = 0.05416  # μm/pixel

#  Burst模式参数
# 540帧 = 60个时间点 × 9帧/时间点 (burst)
FRAMES_PER_TIME_POINT = 9      # 每个时间点的burst帧数
TIME_POINT_INTERVAL_SEC = 60.0  # 时间点间隔（秒）
ACTUAL_FPS = 1.0 / TIME_POINT_INTERVAL_SEC  # 实际帧率（时间点之间）

DISTANCE_HALF_WINDOW_UM = 60     

# 通道配置 (OpenCV: BGR)
CHANNEL_CONFIG = {'cDC': 2, 'CD4_Tn': 1, 'CD8_Tn': 0}

# 预处理核大小
KERNEL_CD4 = 61
KERNEL_CD8 = 101
KERNEL_cDC = 61

# 面积过滤参数
T_CELL_MIN_AREA = 500
T_CELL_MAX_AREA = 60000
cDC_MIN_AREA = 2000
cDC_MAX_AREA = 300000

# 形态学过滤
MIN_SOLIDITY = 0.3
MAX_ECCENTRICITY = 0.98

# Trackpy参数
TRACKING_SEARCH_RANGE = 200
TRACKING_MEMORY = 2
TRACKING_MIN_LENGTH = 3

# =============================================================================
# 输入输出配置
# =============================================================================
INPUT_DIR = r"H:\ZQ_WLlab\movies"
OUTPUT_DIR = r"H:\ZQ_WLlab\movies\track_analysis_output_v4"  
os.makedirs(OUTPUT_DIR, exist_ok=True)

SAMPLE_FRAMES = None
SKIP_EXISTING = False
RUN_MODE = 'batch'

print("="*70)
print("细胞轨迹分析 - 批量处理 V4 (Burst模式修正版)")
print("="*70)
print(f"运行模式: {RUN_MODE}")
print(f"输入目录: {INPUT_DIR}")
print(f"输出目录: {OUTPUT_DIR}")
print(f"CD4 kernel: {KERNEL_CD4}, CD8 kernel: {KERNEL_CD8}, cDC kernel: {KERNEL_cDC}")
print(f"T细胞面积: {T_CELL_MIN_AREA}-{T_CELL_MAX_AREA} px²")
print(f"分析窗口: ±{DISTANCE_HALF_WINDOW_UM}μm")
print(f"像素尺寸: {PIXEL_SIZE_UM} μm/pixel (固定值)")
print(f" Burst模式: 每{FRAMES_PER_TIME_POINT}帧抽取1帧")
print(f" 时间点间隔: {TIME_POINT_INTERVAL_SEC} 秒 (实际帧率: {ACTUAL_FPS:.6f})")
print(f" 540帧视频 → {540//FRAMES_PER_TIME_POINT}个时间点用于tracking")
print("="*70)

# =============================================================================
# 视频元数据读取函数 (V4)
# =============================================================================

def read_video_metadata(video_path):
    """
    读取视频元数据（V4：Burst模式）
    返回每个视频的帧抽取参数
    """
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        cap.release()
        return PIXEL_SIZE_UM, TIME_POINT_INTERVAL_SEC, ACTUAL_FPS, None, None, "failed_to_open"
    
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    avi_fps = cap.get(cv2.CAP_PROP_FPS)
    
    # 计算时间点数
    n_time_points = total_frames // FRAMES_PER_TIME_POINT
    
    # 使用时间点间隔
    fps = ACTUAL_FPS
    frame_interval_min = TIME_POINT_INTERVAL_SEC
    
    try:
        fourcc = int(cap.get(cv2.CAP_PROP_FOURCC))
        fourcc_str = "".join([chr((fourcc >> 8 * i) & 0xFF) for i in range(4)])
    except:
        fourcc_str = "unknown"
    
    cap.release()
    
    video_name = os.path.basename(video_path)
    metadata_info = {
        'video_name': video_name,
        'width': width,
        'height': height,
        'avi_fps': avi_fps,
        'actual_fps': fps,
        'time_point_interval_min': frame_interval_min,
        'pixel_size_um': PIXEL_SIZE_UM,
        'total_frames': total_frames,
        'n_time_points': n_time_points,
        'frames_per_time_point': FRAMES_PER_TIME_POINT,
        'fourcc': fourcc_str,
        'metadata_source': 'manual_corrected_v4_burst',
    }
    
    return PIXEL_SIZE_UM, frame_interval_min, fps, width, height, metadata_info


def get_video_metadata_table(input_dir):
    """扫描目录，汇总元数据"""
    avi_files = sorted(glob.glob(os.path.join(input_dir, "*.avi")))
    metadata_list = []
    
    print(f"\n扫描 {len(avi_files)} 个AVI文件的元数据...")
    
    for video_path in tqdm(avi_files, desc="读取视频元数据"):
        pixel_size_um, frame_interval_min, fps, width, height, info = read_video_metadata(video_path)
        metadata_list.append({
            'video_name': os.path.basename(video_path),
            'width': width, 'height': height,
            'resolution': f"{width}x{height}" if width and height else "unknown",
            'total_frames': info.get('total_frames', 0),
            'n_time_points': info.get('n_time_points', 0),
            'estimated_duration_min': info.get('n_time_points', 0) * TIME_POINT_INTERVAL_SEC / 60,
            'avi_fps': info.get('avi_fps', 'N/A'),
            'actual_fps': fps,
            'time_point_interval_s': frame_interval_min,
            'pixel_size_um': pixel_size_um,
        })
    
    df_metadata = pd.DataFrame(metadata_list)
    
    print(f"\n视频元数据汇总:")
    print(f"  文件数: {len(df_metadata)}")
    print(f"\n  分辨率分布:")
    for res, count in df_metadata['resolution'].value_counts().items():
        print(f"    {res}: {count} 个文件")
    print(f"\n  时间点数统计:")
    print(f"    范围: {df_metadata['n_time_points'].min():.0f} - {df_metadata['n_time_points'].max():.0f}")
    print(f"    均值: {df_metadata['n_time_points'].mean():.1f}")
    print(f"  估算时长:")
    print(f"    范围: {df_metadata['estimated_duration_min'].min():.0f} - {df_metadata['estimated_duration_min'].max():.0f} 分钟")
    print(f"\n   Burst模式配置:")
    print(f"    每{FRAMES_PER_TIME_POINT}帧抽取1帧")
    print(f"    时间点间隔: {TIME_POINT_INTERVAL_SEC} 秒")
    print(f"    AVI中的FPS={df_metadata['avi_fps'].iloc[0] if len(df_metadata)>0 else 'N/A'}已被忽略")
    
    return df_metadata


# =============================================================================
# 图像预处理函数（不变）
# =============================================================================

def preprocess_channel(channel_uint8, kernel_size):
    blurred = cv2.GaussianBlur(channel_uint8, (3, 3), 0)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    tophat = cv2.morphologyEx(blurred, cv2.MORPH_TOPHAT, kernel)
    tophat = cv2.normalize(tophat, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    _, binary = cv2.threshold(tophat, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return binary

def filter_by_area(binary, min_area, max_area):
    binary = morphology.remove_small_objects(binary > 0, min_size=min_area)
    labeled = measure.label(binary)
    for p in measure.regionprops(labeled):
        if p.area > max_area:
            labeled[labeled == p.label] = 0
    binary = labeled > 0
    return binary

def filter_detections(props, min_area, max_area):
    filtered = []
    for p in props:
        if p.area < min_area or p.area > max_area:
            continue
        if p.solidity < MIN_SOLIDITY:
            continue
        if p.eccentricity > MAX_ECCENTRICITY:
            continue
        filtered.append(p)
    return filtered

def get_nearest_cell_center(centers, ref_center):
    if len(centers) == 0:
        return None
    dists = np.linalg.norm(centers - ref_center, axis=1)
    return centers[np.argmin(dists)]


# =============================================================================
# 运动参数计算函数
# =============================================================================

def calculate_motion_metrics(tracks, cdc_positions, half_window_um, pixel_size_um, fps):
    """
    计算运动参数
    fps = 1/60 (时间点之间的帧率)
    """
    if tracks is None or tracks.empty:
        return None
    
    tracks = tracks.reset_index(drop=True)
    
    cDC_df = pd.DataFrame({
        'frame': np.arange(len(cdc_positions)),
        'cdc_y': cdc_positions[:, 0],
        'cdc_x': cdc_positions[:, 1]
    })
    
    merged = tracks.merge(cDC_df, on='frame', how='left')
    
    merged['dist_to_cDC_px'] = np.sqrt(
        (merged['x'] - merged['cdc_x'])**2 + 
        (merged['y'] - merged['cdc_y'])**2
    )
    merged['dist_to_cDC_um'] = merged['dist_to_cDC_px'] * pixel_size_um
    
    half_window_px = half_window_um / pixel_size_um
    merged['in_window'] = (
        (np.abs(merged['x'] - merged['cdc_x']) <= half_window_px) &
        (np.abs(merged['y'] - merged['cdc_y']) <= half_window_px)
    )
    
    merged['dx_px'] = merged.groupby('particle')['x'].diff()
    merged['dy_px'] = merged.groupby('particle')['y'].diff()
    merged['dx_um'] = merged['dx_px'] * pixel_size_um
    merged['dy_um'] = merged['dy_px'] * pixel_size_um
    
    # 速度 = 位移 / 时间间隔（60秒）
    merged['instant_speed_um_per_min'] = np.sqrt(merged['dx_um']**2 + merged['dy_um']**2) * 1
    merged['prev_dist_to_cDC_um'] = merged.groupby('particle')['dist_to_cDC_um'].shift(1)
    merged['v_towards_cDC_um_min'] = (merged['prev_dist_to_cDC_um'] - merged['dist_to_cDC_um']) * 1
    
    merged['chemotaxis_index'] = np.where(
        merged['instant_speed_um_per_min'] > 0,
        merged['v_towards_cDC_um_min'] / merged['instant_speed_um_per_min'],
        0
    )
    
    result = merged[[
        'frame', 'particle', 'y', 'x',
        'dist_to_cDC_um', 'in_window',
        'dx_um', 'dy_um',
        'instant_speed_um_per_min',
        'v_towards_cDC_um_min',
        'chemotaxis_index'
    ]].copy()
    
    return result


# =============================================================================
# 单视频处理函数 (帧抽取)
# =============================================================================

def process_single_video(video_path, sample_frames=None, output_dir=None):
    """
    关键：
    1. 每FRAMES_PER_TIME_POINT帧抽取1帧
    2. 重新编号frame索引（0, 1, 2, ... 对应时间点）
    3. 使用时间点间隔计算速度
    """
    video_name = os.path.basename(video_path)
    
    if output_dir and SKIP_EXISTING:
        video_base = os.path.splitext(video_name)[0]
        summary_pattern = os.path.join(output_dir, f"{video_base}_cell_summary.csv")
        if os.path.exists(summary_pattern):
            print(f"  跳过(已处理): {video_name}")
            return None, None, None, None, None
    
    pixel_size_um, frame_interval_min, fps, width, height, metadata_info = read_video_metadata(video_path)
    
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"  无法打开: {video_name}")
        return None, None, None, None, None
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # 计算实际使用的时间点数
    n_time_points = total_frames // FRAMES_PER_TIME_POINT
    
    # 生成需要读取的帧索引（每个时间点的第1帧）
    # 或者取每个时间点的中间帧：FRAMES_PER_TIME_POINT // 2
    frame_indices = list(range(0, total_frames, FRAMES_PER_TIME_POINT))
    
    if sample_frames:
        frame_indices = frame_indices[:sample_frames]
    
    actual_frames = len(frame_indices)
    estimated_duration_min = actual_frames * frame_interval_min / 60.0
    
    # 读取所有需要的帧
    all_raw_frames = []
    raw_idx = 0
    for target_idx in frame_indices:
        while raw_idx <= target_idx:
            ret, frame = cap.read()
            if not ret:
                break
            raw_idx += 1
        if ret:
            all_raw_frames.append(frame)
    
    cap.release()
    
    if len(all_raw_frames) == 0:
        print(f" 无法读取帧")
        return None, None, None, None, None
    
    # --- 检测初始cDC (第1个时间点) ---
    first_frame = all_raw_frames[0]
    cdc_ch = first_frame[:, :, CHANNEL_CONFIG['cDC']]
    binary_cDC = preprocess_channel(cdc_ch, KERNEL_cDC)
    binary_cDC = filter_by_area(binary_cDC, cDC_MIN_AREA, cDC_MAX_AREA)
    props_cDC = measure.regionprops(measure.label(binary_cDC))
    
    cdc_positions = []
    if props_cDC:
        best_cDC = max(props_cDC, key=lambda p: p.area)
        cdc_positions.append(best_cDC.centroid)
    else:
        binary_cDC_raw = preprocess_channel(cdc_ch, KERNEL_cDC)
        props_raw = measure.regionprops(measure.label(binary_cDC_raw))
        if props_raw:
            best_cDC = max(props_raw, key=lambda p: p.area)
            cdc_positions.append(best_cDC.centroid)
        else:
            cdc_positions.append((first_frame.shape[0]/2, first_frame.shape[1]/2))
    
    all_detections = []
    
    # 逐时间点处理（V4：使用重新编号的time_point_idx）
    for tp_idx in tqdm(range(1, len(all_raw_frames)), desc=f"  {video_name[:40]}", leave=False):
        frame = all_raw_frames[tp_idx]
        
        # --- cDC追踪 ---
        cdc_ch = frame[:, :, CHANNEL_CONFIG['cDC']]
        binary_cDC = preprocess_channel(cdc_ch, KERNEL_cDC)
        binary_cDC = filter_by_area(binary_cDC, cDC_MIN_AREA, cDC_MAX_AREA)
        props_cDC = measure.regionprops(measure.label(binary_cDC))
        
        if props_cDC:
            centers = np.array([p.centroid for p in props_cDC])
            new_cdc = get_nearest_cell_center(centers, cdc_positions[-1])
            cdc_positions.append(new_cdc if new_cdc is not None else cdc_positions[-1])
        else:
            cdc_positions.append(cdc_positions[-1])
        
        # --- CD4检测 ---
        cd4_ch = frame[:, :, CHANNEL_CONFIG['CD4_Tn']]
        binary_CD4 = preprocess_channel(cd4_ch, KERNEL_CD4)
        binary_CD4 = filter_by_area(binary_CD4, T_CELL_MIN_AREA, T_CELL_MAX_AREA)
        props_CD4 = filter_detections(
            measure.regionprops(measure.label(binary_CD4)), 
            T_CELL_MIN_AREA, T_CELL_MAX_AREA
        )
        
        for p in props_CD4:
            all_detections.append({
                'frame': tp_idx,  # 使用时间点索引（非原始帧号）
                'y': p.centroid[0], 'x': p.centroid[1],
                'cell_type': 'CD4_Tn', 'area': p.area
            })
        
        # --- CD8检测 ---
        cd8_ch = frame[:, :, CHANNEL_CONFIG['CD8_Tn']]
        binary_CD8 = preprocess_channel(cd8_ch, KERNEL_CD8)
        binary_CD8 = filter_by_area(binary_CD8, T_CELL_MIN_AREA, T_CELL_MAX_AREA)
        props_CD8 = filter_detections(
            measure.regionprops(measure.label(binary_CD8)), 
            T_CELL_MIN_AREA, T_CELL_MAX_AREA
        )
        
        for p in props_CD8:
            all_detections.append({
                'frame': tp_idx,  # 使用时间点索引
                'y': p.centroid[0], 'x': p.centroid[1],
                'cell_type': 'CD8_Tn', 'area': p.area
            })
    
    cdc_positions = np.array(cdc_positions)
    
    # 追踪
    tracks_data = pd.DataFrame()
    motion_data = pd.DataFrame()
    
    if all_detections:
        df_detections = pd.DataFrame(all_detections)
        
        tp.quiet()
        all_tracks = []
        all_motion = []
        
        for cell_type in ['CD4_Tn', 'CD8_Tn']:
            subset = df_detections[df_detections['cell_type'] == cell_type].copy()
            
            if len(subset) >= TRACKING_MIN_LENGTH:
                try:
                    tracks = tp.link(subset, search_range=TRACKING_SEARCH_RANGE, 
                                    memory=TRACKING_MEMORY)
                    if tracks is not None and not tracks.empty:
                        tracks = tp.filter_stubs(tracks, threshold=TRACKING_MIN_LENGTH)
                        tracks['cell_type'] = cell_type
                        
                        motion = calculate_motion_metrics(
                            tracks, cdc_positions, 
                            DISTANCE_HALF_WINDOW_UM, pixel_size_um, fps
                        )
                        if motion is not None and not motion.empty:
                            motion['cell_type'] = cell_type
                            all_motion.append(motion)
                        
                        all_tracks.append(tracks)
                except tp.SubnetOversizeException:
                    try:
                        tracks = tp.link(subset, search_range=100, memory=TRACKING_MEMORY)
                        if tracks is not None and not tracks.empty:
                            tracks = tp.filter_stubs(tracks, threshold=TRACKING_MIN_LENGTH)
                            tracks['cell_type'] = cell_type
                            motion = calculate_motion_metrics(
                                tracks, cdc_positions, 
                                DISTANCE_HALF_WINDOW_UM, pixel_size_um, fps
                            )
                            if motion is not None and not motion.empty:
                                motion['cell_type'] = cell_type
                                all_motion.append(motion)
                            all_tracks.append(tracks)
                    except:
                        pass
                except Exception:
                    pass
        
        if all_tracks:
            tracks_data = pd.concat(all_tracks, ignore_index=True)
        if all_motion:
            motion_data = pd.concat(all_motion, ignore_index=True)
    
    video_info = {
        'video_name': video_name,
        'pixel_size_um': pixel_size_um,
        'fps': fps,
        'frame_interval_min': frame_interval_min,
        'width': width, 'height': height,
        'total_frames_raw': total_frames,
        'n_time_points': n_time_points,
        'actual_frames_processed': actual_frames,
        'estimated_duration_min': estimated_duration_min,
        'frames_per_time_point': FRAMES_PER_TIME_POINT,
        'metadata_source': metadata_info.get('metadata_source', 'default'),
    }
    
    return tracks_data, motion_data, cdc_positions, actual_frames, video_info


# =============================================================================
# 数据导出函数
# =============================================================================

def export_video_data(motion_data, video_name, video_info, output_dir):
    """导出单个视频的分析数据"""
    video_base = os.path.splitext(video_name)[0]
    
    if motion_data.empty:
        return None
    
    pixel_size_um = video_info['pixel_size_um']
    
    # 逐帧数据
    frame_file = os.path.join(output_dir, f"{video_base}_frame_level.csv")
    export_df = motion_data.copy()
    export_df['video_name'] = video_name
    export_df['pixel_size_um'] = pixel_size_um
    export_df['time_point_interval_min'] = video_info['frame_interval_min']
    
    cols = [
        'video_name', 'cell_type', 'particle', 'frame',
        'y', 'x', 'dist_to_cDC_um', 'in_window',
        'dx_um', 'dy_um', 'instant_speed_um_per_min',
        'v_towards_cDC_um_min', 'chemotaxis_index',
        'pixel_size_um', 'time_point_interval_min'
    ]
    export_df = export_df[[c for c in cols if c in export_df.columns]]
    export_df.to_csv(frame_file, index=False)
    
    # 单细胞汇总
    cell_file = os.path.join(output_dir, f"{video_base}_cell_summary.csv")
    
    cell_summary = motion_data.groupby(['cell_type', 'particle']).agg(
        track_length_frames=('frame', 'count'),
        mean_dist_to_cDC_um=('dist_to_cDC_um', 'mean'),
        min_dist_to_cDC_um=('dist_to_cDC_um', 'min'),
        max_dist_to_cDC_um=('dist_to_cDC_um', 'max'),
        mean_speed_um_per_s=('instant_speed_um_per_min', 'mean'),
        max_speed_um_per_s=('instant_speed_um_per_min', 'max'),
        mean_v_towards_cDC_um_s=('v_towards_cDC_um_min', 'mean'),
        mean_chemotaxis_index=('chemotaxis_index', 'mean'),
        median_chemotaxis_index=('chemotaxis_index', 'median'),
        fraction_in_window=('in_window', 'mean'),
        start_y=('y', 'first'),
        start_x=('x', 'first'),
        end_y=('y', 'last'),
        end_x=('x', 'last')
    ).reset_index()
    
    cell_summary['net_displacement_um'] = np.sqrt(
        (cell_summary['end_x'] - cell_summary['start_x'])**2 +
        (cell_summary['end_y'] - cell_summary['start_y'])**2
    ) * pixel_size_um
    
    cell_summary['video_name'] = video_name
    cell_summary.to_csv(cell_file, index=False)
    
    stats = {
        'video_name': video_name,
        'n_time_points': video_info['n_time_points'],
        'n_CD4_cells': len(cell_summary[cell_summary['cell_type']=='CD4_Tn']),
        'n_CD8_cells': len(cell_summary[cell_summary['cell_type']=='CD8_Tn']),
        'mean_CI_CD4': cell_summary[cell_summary['cell_type']=='CD4_Tn']['mean_chemotaxis_index'].mean() if len(cell_summary[cell_summary['cell_type']=='CD4_Tn'])>0 else np.nan,
        'mean_CI_CD8': cell_summary[cell_summary['cell_type']=='CD8_Tn']['mean_chemotaxis_index'].mean() if len(cell_summary[cell_summary['cell_type']=='CD8_Tn'])>0 else np.nan,
        'mean_speed_CD4': cell_summary[cell_summary['cell_type']=='CD4_Tn']['mean_speed_um_per_min'].mean() if len(cell_summary[cell_summary['cell_type']=='CD4_Tn'])>0 else np.nan,
        'mean_speed_CD8': cell_summary[cell_summary['cell_type']=='CD8_Tn']['mean_speed_um_per_min'].mean() if len(cell_summary[cell_summary['cell_type']=='CD8_Tn'])>0 else np.nan,
        'pixel_size_um': pixel_size_um,
        'time_point_interval_min': video_info['frame_interval_min'],
        'width': video_info.get('width'),
        'height': video_info.get('height'),
    }
    
    return stats


def export_combined_summary(all_stats, all_motion_data, all_video_info, output_dir):
    """导出所有视频的合并汇总"""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    if all_stats:
        summary_df = pd.DataFrame(all_stats)
        summary_file = os.path.join(output_dir, f"all_videos_summary_{timestamp}.csv")
        summary_df.to_csv(summary_file, index=False)
        print(f"\n✓ 视频汇总: {summary_file}")
    
    if all_video_info:
        metadata_df = pd.DataFrame(all_video_info)
        metadata_file = os.path.join(output_dir, f"video_metadata_{timestamp}.csv")
        metadata_df.to_csv(metadata_file, index=False)
        print(f"✓ 视频元数据: {metadata_file}")
    
    if all_motion_data:
        all_frames = []
        all_cells = []
        
        for video_name, motion_data in all_motion_data.items():
            if motion_data is not None and not motion_data.empty:
                frame_data = motion_data.copy()
                frame_data['video_name'] = video_name
                all_frames.append(frame_data)
                
                cell_data = motion_data.groupby(['cell_type', 'particle']).agg(
                    track_length_frames=('frame', 'count'),
                    mean_speed_um_per_s=('instant_speed_um_per_min', 'mean'),
                    mean_v_towards_cDC_um_s=('v_towards_cDC_um_min', 'mean'),
                    mean_chemotaxis_index=('chemotaxis_index', 'mean'),
                    median_chemotaxis_index=('chemotaxis_index', 'median'),
                    fraction_in_window=('in_window', 'mean'),
                ).reset_index()
                cell_data['video_name'] = video_name
                all_cells.append(cell_data)
        
        if all_frames:
            combined_frames = pd.concat(all_frames, ignore_index=True)
            combined_frames = combined_frames.replace([np.inf, -np.inf], np.nan).dropna(subset=['chemotaxis_index'])
            frames_file = os.path.join(output_dir, f"all_videos_frame_level_{timestamp}.csv")
            combined_frames.to_csv(frames_file, index=False)
            print(f"合并逐帧数据: {frames_file} ({len(combined_frames)} 行)")
        
        if all_cells:
            combined_cells = pd.concat(all_cells, ignore_index=True)
            cells_file = os.path.join(output_dir, f"all_videos_cell_summary_{timestamp}.csv")
            combined_cells.to_csv(cells_file, index=False)
            print(f"合并细胞汇总: {cells_file} ({len(combined_cells)} 个细胞)")
        
        return combined_frames, combined_cells
    
    return None, None


# =============================================================================
# 批量处理主函数
# =============================================================================

def batch_process_videos(input_dir, output_dir, sample_frames=None):
    """批量处理"""
    metadata_df = get_video_metadata_table(input_dir)
    metadata_df.to_csv(os.path.join(output_dir, "all_video_metadata_summary.csv"), index=False)
    
    avi_files = sorted(glob.glob(os.path.join(input_dir, "*.avi")))
    
    if not avi_files:
        print(f"\n在 {input_dir} 中未找到AVI文件！")
        return None, None
    
    print(f"\n找到 {len(avi_files)} 个AVI文件")
    print("="*70)
    
    all_stats = []
    all_motion_data = {}
    all_video_info = []
    success_count = 0
    fail_count = 0
    skip_count = 0
    
    for i, video_path in enumerate(avi_files):
        video_name = os.path.basename(video_path)
        print(f"\n[{i+1}/{len(avi_files)}] {video_name}")
        print(f"  文件大小: {os.path.getsize(video_path)/1024/1024:.1f} MB")
        
        try:
            tracks_data, motion_data, cdc_positions, n_frames, video_info = process_single_video(
                video_path, sample_frames, output_dir
            )
            
            if tracks_data is None and motion_data is None:
                skip_count += 1
                continue
            
            if video_info:
                print(f"  分辨率: {video_info.get('width')}x{video_info.get('height')}")
                print(f"  原始帧数: {video_info.get('total_frames_raw', 0)}")
                print(f"  时间点数: {video_info.get('n_time_points', 0)} (≈{video_info.get('estimated_duration_min', 0):.0f}分钟)")
                all_video_info.append(video_info)
            
            if tracks_data is not None and not tracks_data.empty:
                n_cd4 = tracks_data[tracks_data['cell_type']=='CD4_Tn']['particle'].nunique()
                n_cd8 = tracks_data[tracks_data['cell_type']=='CD8_Tn']['particle'].nunique()
                print(f"  CD4={n_cd4}条, CD8={n_cd8}条轨迹 ({n_frames}个时间点)")
                
                if not motion_data.empty:
                    stats = export_video_data(motion_data, video_name, video_info, output_dir)
                    if stats:
                        stats['n_time_points'] = n_frames
                        all_stats.append(stats)
                        all_motion_data[video_name] = motion_data
                
                success_count += 1
            else:
                print(f"  未检测到细胞")
                fail_count += 1
                
        except Exception as e:
            print(f"  错误: {e}")
            import traceback
            traceback.print_exc()
            fail_count += 1
    
    if all_stats:
        print("\n" + "="*70)
        print("导出合并数据...")
        combined_frames, combined_cells = export_combined_summary(all_stats, all_motion_data, all_video_info, output_dir)
    
    print("\n" + "="*70)
    print("批量处理完成!")
    print("="*70)
    print(f"  成功: {success_count} 个视频")
    print(f"  失败: {fail_count} 个视频")
    print(f"  跳过: {skip_count} 个视频")
    print(f"  输出目录: {output_dir}")
    
    if all_stats:
        df_stats = pd.DataFrame(all_stats)
        print(f"\n总体统计 (Burst模式, {FRAMES_PER_TIME_POINT}帧/时间点, 间隔{TIME_POINT_INTERVAL_SEC}分):")
        print(f"  总CD4细胞: {df_stats['n_CD4_cells'].sum()}")
        print(f"  总CD8细胞: {df_stats['n_CD8_cells'].sum()}")
        if not df_stats['mean_speed_CD4'].isna().all():
            print(f"  CD4平均速度: {df_stats['mean_speed_CD4'].mean():.6f} μm/min")
        if not df_stats['mean_speed_CD8'].isna().all():
            print(f"  CD8平均速度: {df_stats['mean_speed_CD8'].mean():.6f} μm/min")
    
    print("="*70)
    
    combined_frames = pd.concat([d for d in all_motion_data.values() if not d.empty], ignore_index=True) if all_motion_data else None
    
    return combined_frames, metadata_df


# =============================================================================
# 主程序
# =============================================================================

if __name__ == "__main__":
    
    print(f"\n开始执行: {RUN_MODE} 模式\n")
    
    if RUN_MODE == 'batch':
        combined_frames, metadata_df = batch_process_videos(
            INPUT_DIR, OUTPUT_DIR, SAMPLE_FRAMES
        )
        
        if metadata_df is not None and not metadata_df.empty:
            print(f"\n{'='*70}")
            print("视频元数据最终汇总")
            print(f"{'='*70}")
            print(f"所有视频统一使用:")
            print(f"  像素尺寸: {PIXEL_SIZE_UM} μm/pixel")
            print(f"  Burst模式: 每{FRAMES_PER_TIME_POINT}帧抽取1帧")
            print(f"  时间点间隔: {TIME_POINT_INTERVAL_SEC} 分")
            print(f"  实际帧率: {ACTUAL_FPS:.6f}")
    
    print("\n完成!")

D:\Softwares\Anaconda\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated and will be removed in a future release
  "class": algorithms.Blowfish,


细胞轨迹分析 - 批量处理 V4 (Burst模式修正版)
运行模式: batch
输入目录: H:\ZQ_WLlab\movies
输出目录: H:\ZQ_WLlab\movies\track_analysis_output_v4
CD4 kernel: 61, CD8 kernel: 101, cDC kernel: 61
T细胞面积: 500-60000 px²
分析窗口: ±60μm
像素尺寸: 0.05416 μm/pixel (固定值)
 Burst模式: 每9帧抽取1帧
 时间点间隔: 60.0 秒 (实际帧率: 0.016667)
 540帧视频 → 60个时间点用于tracking

开始执行: batch 模式


扫描 48 个AVI文件的元数据...


读取视频元数据: 100%|██████████████████████████████████████████████████████████████████| 48/48 [00:04<00:00, 11.00it/s]



视频元数据汇总:
  文件数: 48

  分辨率分布:
    1024x1024: 43 个文件
    1536x1536: 5 个文件

  时间点数统计:
    范围: 4 - 60
    均值: 20.0
  估算时长:
    范围: 4 - 60 分钟

   Burst模式配置:
    每9帧抽取1帧
    时间点间隔: 60.0 秒
    AVI中的FPS=20.0已被忽略

找到 48 个AVI文件

[1/48] 12_Merge_405_Em450_488_Em525_561_Em609_2DSIM-3_1.avi
  文件大小: 12.1 MB


  分辨率: 1536x1536
  原始帧数: 135
  时间点数: 15 (≈15分钟)
  ✓ CD4=16条, CD8=5条轨迹 (15个时间点)

[2/48] 15_Merge_405_Em450_488_Em525_561_Em609_2DSIM-3_1.avi
  文件大小: 2.6 MB


...

  分辨率: 1024x1024
  原始帧数: 90
  时间点数: 10 (≈10分钟)
  ✓ CD4=23条, CD8=13条轨迹 (10个时间点)

[45/48] 6_Merge_405_Em450_488_Em525_561_Em609_2DSIM-3_cDC_CD4_CD8.avi
  文件大小: 4.6 MB


  分辨率: 1024x1024
  原始帧数: 99
  时间点数: 11 (≈11分钟)
  ✓ CD4=1条, CD8=1条轨迹 (11个时间点)

[46/48] 7_Merge_405_Em450_488_Em525_561_Em609_2DSIM-3_cDC_CD4_CD8.avi
  文件大小: 4.9 MB


  分辨率: 1024x1024
  原始帧数: 99
  时间点数: 11 (≈11分钟)
  ✓ CD4=7条, CD8=2条轨迹 (11个时间点)

[47/48] 8_Merge_405_Em450_488_Em525_561_Em609_2DSIM-3_cDC_CD4_CD8.avi
  文件大小: 1.8 MB


  分辨率: 1024x1024
  原始帧数: 36
  时间点数: 4 (≈4分钟)
  ✓ CD4=4条, CD8=4条轨迹 (4个时间点)

[48/48] 9_Merge_405_Em450_488_Em525_561_Em609_2DSIM-3_cDC_CD4_CD8.avi
  文件大小: 5.8 MB


  分辨率: 1024x1024
  原始帧数: 135
  时间点数: 15 (≈15分钟)
  ✓ CD4=8条, CD8=4条轨迹 (15个时间点)

导出合并数据...

✓ 视频汇总: H:\ZQ_WLlab\movies\track_analysis_output_v4\all_videos_summary_20260622_190548.csv
✓ 视频元数据: H:\ZQ_WLlab\movies\track_analysis_output_v4\video_metadata_20260622_190548.csv
✓ 合并逐帧数据: H:\ZQ_WLlab\movies\track_analysis_output_v4\all_videos_frame_level_20260622_190548.csv (13350 行)
✓ 合并细胞汇总: H:\ZQ_WLlab\movies\track_analysis_output_v4\all_videos_cell_summary_20260622_190548.csv (1634 个细胞)

批量处理完成!
  成功: 48 个视频
  失败: 0 个视频
  跳过: 0 个视频
  输出目录: H:\ZQ_WLlab\movies\track_analysis_output_v4

总体统计 (Burst模式, 9帧/时间点, 间隔60.0秒):
  总CD4细胞: 896
  总CD8细胞: 738
  CD4平均速度: 0.039501 μm/s
  CD8平均速度: 0.041860 μm/s

视频元数据最终汇总
所有视频统一使用:
  像素尺寸: 0.05416 μm/pixel
  Burst模式: 每9帧抽取1帧
  时间点间隔: 60.0 秒
  实际帧率: 0.016667

完成!


In [ ]:
# =============================================================================
# Napari轨迹可视化 + 视频导出模块 - 仅导出版
# 只导出MP4视频，不打开Napari窗口
# =============================================================================

import os
import glob
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# 配置
# =============================================================================
DATA_DIR = r"H:\ZQ_WLlab\movies\track_analysis_output_v4"
VIDEO_INPUT_DIR = r"H:\ZQ_WLlab\movies"
OUTPUT_DIR = r"H:\ZQ_WLlab\movies\track_analysis_output_v4\visualization"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 通道配置
CHANNEL_CONFIG = {'cDC': 2, 'CD4_Tn': 1, 'CD8_Tn': 0}

FRAMES_PER_TIME_POINT = 9

# 可视化参数
TRACK_TAIL_LENGTH = 5
TRACK_LINE_WIDTH = 2
T_CELL_MARKER_SIZE = 8
VIDEO_FPS = 5

# =============================================================================
# 加载数据
# =============================================================================

def load_tracks_for_video(video_name):
    """加载指定视频的追踪数据"""
    frame_files = sorted(glob.glob(os.path.join(DATA_DIR, "all_videos_frame_level_*.csv")))
    if not frame_files:
        print("未找到逐帧数据文件！")
        return None
    
    df_frame = pd.read_csv(frame_files[-1])
    video_data = df_frame[df_frame['video_name'] == video_name].copy()
    
    if video_data.empty:
        video_base = os.path.splitext(video_name)[0]
        single_file = os.path.join(DATA_DIR, f"{video_base}_frame_level.csv")
        if os.path.exists(single_file):
            video_data = pd.read_csv(single_file)
        else:
            return None
    
    tracks_df = video_data[['particle', 'frame', 'y', 'x', 'cell_type']].dropna()
    return tracks_df


# =============================================================================
# 导出视频
# =============================================================================

def export_trajectory_video_opencv(video_name, tracks_df, output_dir=None):
    """使用OpenCV导出轨迹可视化视频"""
    video_path = os.path.join(VIDEO_INPUT_DIR, video_name)
    
    if not os.path.exists(video_path):
        print(f" 视频文件不存在: {video_name}")
        return None
    
    video_base = os.path.splitext(video_name)[0]
    output_path = os.path.join(output_dir or OUTPUT_DIR, f"{video_base}_trajectories.mp4")
    
    # 读取视频
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    all_frames = []
    for _ in range(total_frames):
        ret, frame = cap.read()
        if ret:
            all_frames.append(frame)
    cap.release()
    
    # 抽取时间点帧
    time_point_frames = [all_frames[i] for i in range(0, len(all_frames), FRAMES_PER_TIME_POINT)]
    n_time_points = len(time_point_frames)
    height, width = time_point_frames[0].shape[:2]
    
    # 追踪数据
    cd4_tracks = tracks_df[tracks_df['cell_type'] == 'CD4_Tn']
    cd8_tracks = tracks_df[tracks_df['cell_type'] == 'CD8_Tn']
    
    # 创建视频写入器
    output_width, output_height = width * 2, height * 2
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video_writer = cv2.VideoWriter(output_path, fourcc, VIDEO_FPS, (output_width, output_height))
    
    if not video_writer.isOpened():
        output_path = output_path.replace('.mp4', '.avi')
        fourcc = cv2.VideoWriter_fourcc(*'XVID')
        video_writer = cv2.VideoWriter(output_path, fourcc, VIDEO_FPS, (output_width, output_height))
    
    for tp_idx in range(n_time_points):
        frame = time_point_frames[tp_idx]
        cdc_ch = frame[..., CHANNEL_CONFIG['cDC']]
        cd4_ch = frame[..., CHANNEL_CONFIG['CD4_Tn']]
        cd8_ch = frame[..., CHANNEL_CONFIG['CD8_Tn']]
        
        output_frame = np.zeros((output_height, output_width, 3), dtype=np.uint8)
        
        # 图1: cDC (左上)
        cdc_color = cv2.cvtColor(cdc_ch, cv2.COLOR_GRAY2BGR)
        cdc_color[..., 0], cdc_color[..., 1] = 0, 0
        cdc_color[..., 2] = cdc_ch
        output_frame[0:height, 0:width] = cdc_color
        
        # 图2: CD4 + 轨迹 (右上)
        cd4_color = cv2.cvtColor(cd4_ch, cv2.COLOR_GRAY2BGR)
        cd4_color[..., 0], cd4_color[..., 2] = 0, 0
        cd4_color[..., 1] = cd4_ch
        
        history_cd4 = cd4_tracks[cd4_tracks['frame'] <= tp_idx]
        for _, track in history_cd4.groupby('particle'):
            if len(track) > 1:
                pts = track.tail(TRACK_TAIL_LENGTH)[['x', 'y']].values.astype(np.int32)
                for j in range(len(pts) - 1):
                    cv2.line(cd4_color, tuple(pts[j]), tuple(pts[j+1]), (0, 255, 0), TRACK_LINE_WIDTH)
        
        current_cd4 = cd4_tracks[cd4_tracks['frame'] == tp_idx]
        for _, cell in current_cd4.iterrows():
            cv2.circle(cd4_color, (int(cell['x']), int(cell['y'])), T_CELL_MARKER_SIZE*2, (0, 255, 0), -1)
        
        output_frame[0:height, width:width*2] = cd4_color
        
        # 图3: CD8 + 轨迹 (左下)
        cd8_color = cv2.cvtColor(cd8_ch, cv2.COLOR_GRAY2BGR)
        cd8_color[..., 1], cd8_color[..., 2] = 0, 0
        cd8_color[..., 0] = cd8_ch
        
        history_cd8 = cd8_tracks[cd8_tracks['frame'] <= tp_idx]
        for _, track in history_cd8.groupby('particle'):
            if len(track) > 1:
                pts = track.tail(TRACK_TAIL_LENGTH)[['x', 'y']].values.astype(np.int32)
                for j in range(len(pts) - 1):
                    cv2.line(cd8_color, tuple(pts[j]), tuple(pts[j+1]), (255, 255, 0), TRACK_LINE_WIDTH)
        
        current_cd8 = cd8_tracks[cd8_tracks['frame'] == tp_idx]
        for _, cell in current_cd8.iterrows():
            cv2.circle(cd8_color, (int(cell['x']), int(cell['y'])), T_CELL_MARKER_SIZE*2, (255, 255, 0), -1)
        
        output_frame[height:height*2, 0:width] = cd8_color
        
        # 图4: 合并RGB + 轨迹 (右下)
        rgb_merged = np.zeros((height, width, 3), dtype=np.uint8)
        for ch_idx, ch in enumerate([cdc_ch, cd4_ch, cd8_ch]):
            ch_norm = ch.astype(float)
            if ch_norm.max() > 0:
                ch_norm = ch_norm / ch_norm.max() * 255
            rgb_merged[..., 2-ch_idx] = ch_norm.astype(np.uint8)
        
        for _, track in history_cd4.groupby('particle'):
            if len(track) > 1:
                pts = track.tail(TRACK_TAIL_LENGTH)[['x', 'y']].values.astype(np.int32)
                for j in range(len(pts) - 1):
                    cv2.line(rgb_merged, tuple(pts[j]), tuple(pts[j+1]), (0, 255, 0), TRACK_LINE_WIDTH)
        
        for _, track in history_cd8.groupby('particle'):
            if len(track) > 1:
                pts = track.tail(TRACK_TAIL_LENGTH)[['x', 'y']].values.astype(np.int32)
                for j in range(len(pts) - 1):
                    cv2.line(rgb_merged, tuple(pts[j]), tuple(pts[j+1]), (255, 255, 0), TRACK_LINE_WIDTH)
        
        output_frame[height:height*2, width:width*2] = rgb_merged
        
        # 标注
        cv2.putText(output_frame, f"t={tp_idx+1}/{n_time_points} ({tp_idx} min)", 
                   (10, output_height-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1)
        for label, x, y in [("cDC", 10, 20), ("CD4", width+10, 20), 
                            ("CD8", 10, height+20), ("Merged", width+10, height+20)]:
            cv2.putText(output_frame, label, (x, y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 1)
        
        video_writer.write(output_frame)
    
    video_writer.release()
    return output_path


# =============================================================================
# 批量导出
# =============================================================================

def batch_export_videos(skip_existing=True):
    """批量导出所有视频的轨迹可视化"""
    frame_files = sorted(glob.glob(os.path.join(DATA_DIR, "all_videos_frame_level_*.csv")))
    if not frame_files:
        print("未找到追踪数据文件！")
        return
    
    df_all = pd.read_csv(frame_files[-1])
    video_names = sorted(df_all['video_name'].unique())
    
    print(f"\n{'='*60}")
    print(f"批量导出轨迹视频 - 共 {len(video_names)} 个视频")
    print(f"{'='*60}\n")
    
    success = 0
    skipped = 0
    
    for i, video_name in enumerate(video_names):
        video_base = os.path.splitext(video_name)[0]
        output_path = os.path.join(OUTPUT_DIR, f"{video_base}_trajectories.mp4")
        
        if skip_existing and os.path.exists(output_path):
            print(f"[{i+1:2d}/{len(video_names)}] {video_name[:50]}... 已存在")
            skipped += 1
            continue
        
        print(f"[{i+1:2d}/{len(video_names)}] {video_name[:50]}...", end=" ")
        
        tracks_df = load_tracks_for_video(video_name)
        if tracks_df is None or tracks_df.empty:
            print("无数据")
            continue
        
        output = export_trajectory_video_opencv(video_name, tracks_df)
        if output:
            print("✓")
            success += 1
        else:
            print("✗")
    
    print(f"\n{'='*60}")
    print(f"完成! 成功: {success}, 跳过: {skipped}")
    print(f"输出目录: {OUTPUT_DIR}")
    print(f"{'='*60}")


# =============================================================================
# 主程序
# =============================================================================

if __name__ == "__main__":
    print("="*60)
    print("轨迹视频批量导出工具")
    print("="*60)
    batch_export_videos(skip_existing=True)
    print("\n完成!")

轨迹视频批量导出工具

批量导出轨迹视频 - 共 48 个视频

[ 1/48] 12_Merge_405_Em450_488_Em525_561_Em609_2DSIM-3_1.a... ⏭ 已存在
[ 2/48] 15_Merge_405_Em450_488_Em525_561_Em609_2DSIM-3_1.a... ⏭ 已存在
[ 3/48] 17_Merge_405_Em450_488_Em525_561_Em609_2DSIM-3_1.a... ⏭ 已存在
[ 4/48] 18_Merge_405_Em450_488_Em525_561_Em609_2DSIM-3_1.a... ⏭ 已存在
[ 5/48] 19_Merge_405_Em450_488_Em525_561_Em609_2DSIM-3_1.a... ⏭ 已存在
[ 6/48] 21_Merge_405_Em450_488_Em525_561_Em609_2DSIM-3_1.a... ⏭ 已存在
[ 7/48] 22_Merge_405_Em450_488_Em525_561_Em609_2DSIM-3_1.a... ⏭ 已存在
[ 8/48] 29_Merge_405_Em450_488_Em525_561_Em609_2DSIM-3_1.a... ⏭ 已存在
[ 9/48] 2_0_Merge405-488-561__2DSIM-3_cDC_CD4Tn_CD8Tn.avi... ⏭ 已存在
[10/48] 2_14_Pos_Merge_405_Em450_488_Em525_561_Em609_2DSIM... ⏭ 已存在
[11/48] 2_15_Pos_Merge_405_Em450_488_Em525_561_Em609_2DSIM... ⏭ 已存在
[12/48] 2_16_Pos_Merge_405_Em450_488_Em525_561_Em609_2DSIM... ⏭ 已存在
[13/48] 2_1_Pos_Merge_405_Em450_488_Em525_561_Em609_2DSIM-... ⏭ 已存在
[14/48] 2_2_Pos_Merge_405_Em450_488_Em525_561_Em609_2DSIM-... ⏭ 已存在
[15/48] 2_3_Pos_